# Phân tích Benchmark — Dự báo PM2.5

**6 mô hình** · **5 tầm nhìn** · **3 block split** · **2 vùng**

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#fafafa',
    'axes.edgecolor': '#cccccc', 'axes.labelcolor': '#222222',
    'text.color': '#222222', 'xtick.color': '#444444', 'ytick.color': '#444444',
    'grid.color': '#e0e0e0', 'grid.alpha': 0.7,
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'figure.titlesize': 13, 'figure.titleweight': 'bold',
    'legend.facecolor': 'white', 'legend.edgecolor': '#cccccc', 'legend.fontsize': 8,
    'figure.dpi': 120
})

MC = {
    'XGBoost': '#d95f02', 'XLinear': '#1b9e77', 'iTransformer': '#7570b3',
    'ESTGCN': '#e7298a', 'ST-XLinear': '#66a61e', 'Ensemble': '#e6ab02'
}
MM = {'XGBoost':'o','XLinear':'s','iTransformer':'^','ESTGCN':'D','ST-XLinear':'P','Ensemble':'*'}
SHORT = {'iTransformer':'iTrans','ST-XLinear':'ST-XL'}
def sn(m): return SHORT.get(m, m)

SAVE_DIR = 'figures'
os.makedirs(SAVE_DIR, exist_ok=True)
def savefig(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, f'{name}.png'), dpi=250, bbox_inches='tight', facecolor='white')

with open('benchmark_metrics.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

HORIZONS = raw['horizons']; MODELS = raw['models']; METRICS = raw['metrics']
BLOCKS = list(raw['data'].keys()); REGIONS = list(raw['data'][BLOCKS[0]].keys())
H_NUMS = [1,3,6,12,24]

rows = []
for blk in BLOCKS:
    for reg in REGIONS:
        for met in METRICS + (['train_time'] if 'train_time' in raw['data'][blk][reg] else []):
            for mdl in MODELS:
                vals = raw['data'][blk][reg][met].get(mdl, [None]*5)
                for hi, hl in enumerate(HORIZONS):
                    rows.append(dict(block=blk, region=reg, model=mdl, horizon=hl,
                                     horizon_h=H_NUMS[hi], metric=met,
                                     value=float(vals[hi]) if vals[hi] is not None else np.nan))
df = pd.DataFrame(rows)

def avg_reg(blk, met):
    s = df[(df['block']==blk)&(df['metric']==met)]
    return s.groupby(['model','horizon_h'])['value'].mean().unstack()

def gv(blk, reg, met, mdl, h):
    s = df[(df['block']==blk)&(df['region']==reg)&(df['metric']==met)&(df['model']==mdl)&(df['horizon_h']==h)]
    return s['value'].values[0] if len(s)>0 else np.nan

print(f'Loaded {len(df)} records')

---
## 1. Tất cả chỉ số theo tầm nhìn dự báo (MAE · RMSE · R² · MAPE)

In [ ]:
for met in ['MAE', 'RMSE', 'R2', 'MAPE']:
    fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=True)
    fig.suptitle(f'{met} theo tầm nhìn dự báo', fontsize=14, fontweight='bold', y=1.0)
    
    for col_i, blk in enumerate(BLOCKS):
        for row_i, reg in enumerate(REGIONS):
            ax = axes[row_i, col_i]
            sub = df[(df['block']==blk)&(df['region']==reg)&(df['metric']==met)]
            for mdl in MODELS:
                ms = sub[sub['model']==mdl].sort_values('horizon_h')
                ax.plot(ms['horizon_h'], ms['value'], marker=MM[mdl], color=MC[mdl],
                        linewidth=1.8, markersize=6, alpha=0.9)
                for _, r in ms.iterrows():
                    fmt = f'{r["value"]:.2f}' if met == 'R2' else f'{r["value"]:.1f}'
                    ax.annotate(fmt, (r['horizon_h'], r['value']),
                               textcoords='offset points', xytext=(0,7), fontsize=5.5,
                               color=MC[mdl], ha='center', fontweight='bold')
            ax.set_title(f'{blk} — {reg.capitalize()}', fontsize=10)
            ax.set_xticks(H_NUMS)
            ax.set_xticklabels([f'T+{h}' for h in H_NUMS], fontsize=8)
            ax.grid(True, linestyle='--', alpha=0.4)
            if col_i == 0:
                unit = ' (µg/m³)' if met in ('MAE','RMSE') else (' (%)' if met=='MAPE' else '')
                ax.set_ylabel(f'{met}{unit}', fontsize=9)
            if row_i == 1: ax.set_xlabel('Tầm nhìn dự báo', fontsize=9)
    
    handles = [Line2D([0],[0], color=MC[m], marker=MM[m], linewidth=1.5, label=m) for m in MODELS]
    fig.legend(handles=handles, loc='lower center', ncol=6, fontsize=9,
              frameon=True, fancybox=True, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0, 0.04, 1, 0.98])
    savefig(fig, f'metric_{met.lower()}_vs_horizon')
    plt.show()

---
## 2. R² Heatmap — Tất cả blocks × vùng

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for col_i, blk in enumerate(BLOCKS):
    for row_i, reg in enumerate(REGIONS):
        ax = axes[row_i, col_i]
        sub = df[(df['block']==blk)&(df['region']==reg)&(df['metric']=='R2')]
        pivot = sub.pivot_table(index='model', columns='horizon', values='value')
        pivot = pivot[HORIZONS]
        pivot = pivot.loc[[m for m in MODELS if m in pivot.index]]
        
        im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
        ax.set_xticks(range(len(HORIZONS)))
        ax.set_xticklabels(HORIZONS, fontsize=8)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([sn(m) for m in pivot.index], fontsize=8)
        ax.set_title(f'{blk} — {reg.capitalize()}', fontsize=10)
        
        for i in range(len(pivot.index)):
            for j in range(len(HORIZONS)):
                v = pivot.values[i, j]
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=8, fontweight='bold',
                        color='white' if v < 0.45 else 'black')

fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.94, 0.15, 0.012, 0.7])
fig.colorbar(im, cax=cbar_ax, label='R²')
fig.tight_layout(rect=[0, 0, 0.93, 1])
savefig(fig, 'r2_heatmap_all')
plt.show()

---
## 3. Khoảng cách MAE: T+1 vs T+24

In [ ]:
# Tìm giá trị max toàn cục để đặt chung trục Y
global_max_mae24 = 0
for blk in BLOCKS:
    for reg in REGIONS:
        sub = df[(df['block']==blk)&(df['region']==reg)&(df['metric']=='MAE')&(df['horizon_h']==24)]
        mx = sub['value'].max()
        if mx > global_max_mae24: global_max_mae24 = mx
ylim_gap = global_max_mae24 * 1.25

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)

for col_i, blk in enumerate(BLOCKS):
    for row_i, reg in enumerate(REGIONS):
        ax = axes[row_i, col_i]
        sub = df[(df['block']==blk)&(df['region']==reg)&(df['metric']=='MAE')]
        x = np.arange(len(MODELS))
        w = 0.35
        
        v1  = [sub[(sub['model']==m)&(sub['horizon_h']==1)]['value'].values[0] for m in MODELS]
        v24 = [sub[(sub['model']==m)&(sub['horizon_h']==24)]['value'].values[0] for m in MODELS]
        
        ax.bar(x-w/2, v1, w, color=[MC[m] for m in MODELS], alpha=0.9)
        ax.bar(x+w/2, v24, w, color=[MC[m] for m in MODELS], alpha=0.4,
               edgecolor=[MC[m] for m in MODELS], linewidth=1.2)
        
        for i in range(len(MODELS)):
            ax.text(x[i]-w/2, v1[i]*0.5, f'{v1[i]:.1f}', ha='center', va='center',
                   fontsize=6, color='white', fontweight='bold')
            ax.text(x[i]+w/2, v24[i]*0.5, f'{v24[i]:.1f}', ha='center', va='center',
                   fontsize=6, color='#333', fontweight='bold')
            delta = v24[i] - v1[i]
            ax.text(x[i], max(v1[i],v24[i]) + ylim_gap*0.02,
                   f'\u0394{delta:+.1f}', ha='center', fontsize=6,
                   color='#c0392b', fontweight='bold')
        
        ax.set_xticks(x)
        ax.set_xticklabels([sn(m) for m in MODELS], rotation=35, ha='right', fontsize=8)
        ax.set_title(f'{blk} — {reg.capitalize()}', fontsize=10)
        if col_i == 0: ax.set_ylabel('MAE (µg/m³)', fontsize=9)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)

fig.legend(handles=[Patch(facecolor='#888',alpha=0.9,label='T+1'),
                    Patch(facecolor='#888',alpha=0.35,edgecolor='#888',linewidth=1.2,label='T+24')],
          loc='lower center', ncol=2, fontsize=9,
          frameon=True, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.03, 1, 1])
savefig(fig, 'mae_t1_vs_t24_gap')
plt.show()

---
## 4. Box plot RMSE — Từng block × từng horizon

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(22, 14), sharey=True)

for row_i, blk in enumerate(BLOCKS):
    for col_i, h in enumerate(H_NUMS):
        ax = axes[row_i, col_i]
        box_data, colors = [], []
        for mdl in MODELS:
            vals = df[(df['block']==blk)&(df['model']==mdl)
                      &(df['metric']=='RMSE')&(df['horizon_h']==h)]['value'].tolist()
            box_data.append(vals)
            colors.append(MC[mdl])
        
        bp = ax.boxplot(box_data, patch_artist=True, widths=0.55,
                       medianprops=dict(color='black', linewidth=1.5),
                       flierprops=dict(marker='o', markersize=4))
        for patch, c in zip(bp['boxes'], colors):
            patch.set_facecolor(c); patch.set_alpha(0.65)
        
        for i, (data, c) in enumerate(zip(box_data, colors)):
            jit = np.random.normal(i+1, 0.06, size=len(data))
            ax.scatter(jit, data, color=c, alpha=0.7, s=22, zorder=3,
                      edgecolors='#333', linewidths=0.3)
            med = np.median(data)
            ax.text(i+1, med, f'{med:.1f}', ha='center', va='bottom',
                   fontsize=6, color='black', fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.12', facecolor='white',
                            alpha=0.75, edgecolor='none'))
        
        ax.set_xticklabels([sn(m) for m in MODELS], rotation=40, ha='right', fontsize=7)
        ax.set_title(f'{blk} — T+{h}h', fontsize=9)
        if col_i == 0: ax.set_ylabel('RMSE (µg/m³)', fontsize=9)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)

legend_els = [Patch(facecolor=MC[m], alpha=0.7, edgecolor='#333', label=m) for m in MODELS]
fig.legend(handles=legend_els, loc='lower center', ncol=6, fontsize=9,
          frameon=True, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.03, 1, 1])
savefig(fig, 'rmse_boxplot_all_blocks')
plt.show()

---
## 5. Radar Chart — So sánh đa metric tại T+1

In [ ]:
def norm_radar(blk, reg):
    raw_d = {mdl: {met: gv(blk,reg,met,mdl,1) for met in ['MAE','RMSE','R2','MAPE']} for mdl in MODELS}
    all_v = {met: [raw_d[m][met] for m in MODELS] for met in ['MAE','RMSE','R2','MAPE']}
    result = {}
    for mdl in MODELS:
        sc = []
        for met in ['MAE','RMSE','MAPE']:
            mn, mx = min(all_v[met]), max(all_v[met])
            sc.append(1 - (raw_d[mdl][met]-mn)/(mx-mn+1e-9))
        mn, mx = min(all_v['R2']), max(all_v['R2'])
        sc.insert(2, (raw_d[mdl]['R2']-mn)/(mx-mn+1e-9))
        result[mdl] = sc
    return result

cats = ['MAE\n(lower=better)', 'RMSE\n(lower=better)', 'R²\n(higher=better)', 'MAPE\n(lower=better)']
N = len(cats)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(2, 3, figsize=(18, 11), subplot_kw=dict(polar=True))

for col_i, blk in enumerate(BLOCKS):
    for row_i, reg in enumerate(REGIONS):
        ax = axes[row_i, col_i]
        normed = norm_radar(blk, reg)
        for mdl in MODELS:
            vals = normed[mdl] + normed[mdl][:1]
            ax.plot(angles, vals, color=MC[mdl], linewidth=1.8, alpha=0.85)
            ax.fill(angles, vals, color=MC[mdl], alpha=0.06)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(cats, fontsize=7)
        ax.set_ylim(0, 1.1)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(['0.25','0.50','0.75','1.00'], fontsize=6, color='#888')
        ax.set_title(f'{blk} — {reg.capitalize()}', pad=18, fontsize=10)

legend_els = [Line2D([0],[0], color=MC[m], linewidth=2, label=m) for m in MODELS]
fig.legend(handles=legend_els, loc='lower center', ncol=6, fontsize=9,
          frameon=True, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.04, 1, 1])
savefig(fig, 'radar_t1_all')
plt.show()

---
## 6. Thời gian huấn luyện

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
tt = raw['training_times']

for ax_i, blk in enumerate(BLOCKS):
    ax = axes[ax_i]
    x = np.arange(len(MODELS))
    vals = [tt[blk].get(m,0)/60 for m in MODELS]
    bars = ax.bar(x, vals, color=[MC[m] for m in MODELS], alpha=0.85,
                 edgecolor='#666', linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v*0.5,
               f'{v:.1f}', ha='center', va='center', fontsize=8,
               fontweight='bold', color='white' if v > 3 else '#333')
    ax.set_xticks(x)
    ax.set_xticklabels([sn(m) for m in MODELS], rotation=30, ha='right', fontsize=8)
    ax.set_title(blk, fontsize=10)
    if ax_i == 0: ax.set_ylabel('Thời gian (phút)', fontsize=9)
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)

handles = [Patch(facecolor=MC[m], alpha=0.85, label=m) for m in MODELS]
fig.legend(handles=handles, loc='lower center', ncol=6, fontsize=9,
          frameon=True, bbox_to_anchor=(0.5, -0.05))
fig.tight_layout(rect=[0, 0.05, 1, 1])
savefig(fig, 'training_time')
plt.show()

---
## 7. Ensemble vs Best Individual — Δ MAE

In [ ]:
# Tìm global range cho trục Y chung
all_deltas = []
for blk in BLOCKS:
    for reg in REGIONS:
        for h in H_NUMS:
            sub = df[(df['block']==blk)&(df['region']==reg)&(df['metric']=='MAE')&(df['horizon_h']==h)]
            ens = sub[sub['model']=='Ensemble']['value'].values
            best_v = sub[sub['model']!='Ensemble']['value'].min()
            if len(ens)>0: all_deltas.append(ens[0]-best_v)
yabs = max(abs(d) for d in all_deltas) * 1.35

fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=True)

for col_i, blk in enumerate(BLOCKS):
    for row_i, reg in enumerate(REGIONS):
        ax = axes[row_i, col_i]
        deltas, xlabels, bar_colors = [], [], []
        for h in H_NUMS:
            sub = df[(df['block']==blk)&(df['region']==reg)&(df['metric']=='MAE')&(df['horizon_h']==h)]
            ens = sub[sub['model']=='Ensemble']['value'].values
            best_v = sub[sub['model']!='Ensemble']['value'].min()
            if len(ens)>0:
                d = ens[0] - best_v
                deltas.append(d)
                xlabels.append(f'T+{h}')
                bar_colors.append('#27ae60' if d <= 0 else '#c0392b')
        
        bars = ax.bar(xlabels, deltas, color=bar_colors, alpha=0.8, edgecolor='#666', linewidth=0.5)
        ax.set_ylim(-yabs, yabs)
        
        for bar, d in zip(bars, deltas):
            # Đặt label bên trong bar
            y_pos = d * 0.5 if abs(d) > yabs*0.08 else (d + yabs*0.04 if d >= 0 else d - yabs*0.04)
            ax.text(bar.get_x()+bar.get_width()/2, y_pos,
                   f'{d:+.2f}', ha='center', va='center', fontsize=7,
                   fontweight='bold', color='white' if abs(d) > yabs*0.08 else '#333')
        
        ax.axhline(y=0, color='#888', linewidth=1, linestyle='-', alpha=0.6)
        ax.set_title(f'{blk} — {reg.capitalize()}', fontsize=10)
        if col_i == 0: ax.set_ylabel('\u0394 MAE (µg/m³)', fontsize=9)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)

fig.legend(handles=[Patch(facecolor='#27ae60',label='Ensemble tốt hơn'),
                    Patch(facecolor='#c0392b',label='Ensemble kém hơn')],
          loc='lower center', ncol=2, fontsize=9,
          frameon=True, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.04, 1, 1])
savefig(fig, 'ensemble_delta')
plt.show()

---
## 8. Chênh lệch hiệu suất North vs South

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=True)

for row_i, met in enumerate(['MAE', 'RMSE']):
    for col_i, blk in enumerate(BLOCKS):
        ax = axes[row_i, col_i]
        for mdl in MODELS:
            nv = df[(df['block']==blk)&(df['region']=='north')&(df['model']==mdl)&(df['metric']==met)].sort_values('horizon_h')['value'].values
            sv = df[(df['block']==blk)&(df['region']=='south')&(df['model']==mdl)&(df['metric']==met)].sort_values('horizon_h')['value'].values
            if len(nv)==5 and len(sv)==5:
                gap = sv - nv
                ax.plot(H_NUMS, gap, marker=MM[mdl], color=MC[mdl], linewidth=1.8, markersize=6)
                ax.annotate(f'{gap[-1]:.1f}', (H_NUMS[-1], gap[-1]),
                           textcoords='offset points', xytext=(8,0), fontsize=6,
                           color=MC[mdl], fontweight='bold')
        
        ax.axhline(y=0, color='#888', linestyle=':', alpha=0.5)
        ax.set_title(f'{met} gap — {blk}', fontsize=10)
        ax.set_xticks(H_NUMS)
        ax.set_xticklabels([f'T+{h}' for h in H_NUMS], fontsize=8)
        ax.grid(True, linestyle='--', alpha=0.3)
        if col_i == 0: ax.set_ylabel(f'\u0394 {met} (South\u2212North)', fontsize=9)
        if row_i == 1: ax.set_xlabel('Tầm nhìn dự báo', fontsize=9)

fig.legend(handles=[Line2D([0],[0],color=MC[m],marker=MM[m],linewidth=1.5,label=m) for m in MODELS],
          loc='lower center', ncol=6, fontsize=9,
          frameon=True, bbox_to_anchor=(0.5, -0.01))
fig.tight_layout(rect=[0, 0.04, 1, 1])
savefig(fig, 'north_south_gap')
plt.show()

---
## 9. Tốc độ suy giảm MAE (Degradation Slope)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

# Tính global max slope
all_slopes = []
for blk in BLOCKS:
    piv = avg_reg(blk, 'MAE')
    for mdl in MODELS:
        if mdl in piv.index:
            all_slopes.append(np.polyfit(H_NUMS, piv.loc[mdl].values, 1)[0])
xmax_sl = max(all_slopes) * 1.2

for ax_i, blk in enumerate(BLOCKS):
    ax = axes[ax_i]
    slopes = {}
    piv = avg_reg(blk, 'MAE')
    for mdl in MODELS:
        if mdl in piv.index:
            slopes[mdl] = np.polyfit(H_NUMS, piv.loc[mdl].values, 1)[0]
    
    sorted_m = sorted(slopes, key=lambda m: slopes[m])
    vs = [slopes[m] for m in sorted_m]
    cs = [MC[m] for m in sorted_m]
    
    bars = ax.barh(range(len(sorted_m)), vs, color=cs, alpha=0.85,
                  edgecolor='#666', linewidth=0.5, height=0.55)
    ax.set_xlim(0, xmax_sl)
    ax.set_yticks(range(len(sorted_m)))
    ax.set_yticklabels(sorted_m, fontsize=9)
    ax.set_title(blk, fontsize=10)
    if ax_i == 0: ax.set_xlabel('Slope (µg/m³ per hour)', fontsize=9)
    ax.grid(True, axis='x', linestyle='--', alpha=0.3)
    
    for bar, v in zip(bars, vs):
        ax.text(v*0.5, bar.get_y()+bar.get_height()/2,
               f'{v:.3f}', va='center', ha='center', fontsize=8,
               color='white', fontweight='bold')

fig.tight_layout()
savefig(fig, 'degradation_slope')
plt.show()

---
## 10. So sánh chiến lược Block Split (T+1 · T+6 · T+24)

In [ ]:
for met in ['MAE', 'RMSE', 'R2', 'MAPE']:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle(f'{met} — So sánh Block Split', fontsize=12, fontweight='bold', y=1.0)
    
    for col_i, h in enumerate([1, 6, 24]):
        ax = axes[col_i]
        for mdl in MODELS:
            vals = [df[(df['block']==blk)&(df['model']==mdl)&(df['metric']==met)
                       &(df['horizon_h']==h)]['value'].mean() for blk in BLOCKS]
            ax.plot(range(len(BLOCKS)), vals, marker=MM[mdl], color=MC[mdl],
                   linewidth=1.8, markersize=7)
            for xi, v in enumerate(vals):
                fmt = f'{v:.2f}' if met=='R2' else f'{v:.1f}'
                ax.annotate(fmt, (xi, v), textcoords='offset points', xytext=(0,7),
                          fontsize=6, color=MC[mdl], ha='center', fontweight='bold')
        
        ax.set_xticks(range(len(BLOCKS)))
        ax.set_xticklabels(BLOCKS, fontsize=9)
        ax.set_title(f'T+{h}h', fontsize=10)
        ax.grid(True, linestyle='--', alpha=0.3)
        if col_i == 0:
            unit = ' (µg/m³)' if met in ('MAE','RMSE') else (' (%)' if met=='MAPE' else '')
            ax.set_ylabel(f'{met}{unit}', fontsize=9)
    
    fig.legend(handles=[Line2D([0],[0],color=MC[m],marker=MM[m],linewidth=1.5,label=m) for m in MODELS],
              loc='lower center', ncol=6, fontsize=9,
              frameon=True, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout(rect=[0, 0.06, 1, 0.98])
    savefig(fig, f'block_compare_{met.lower()}')
    plt.show()

---
## 11. Bảng tổng hợp — Block 7 (trung bình 2 vùng)

In [ ]:
rows_s = []
for mdl in MODELS:
    row = {'Model': mdl}
    for h in H_NUMS:
        for met in ['MAE','RMSE','R2','MAPE']:
            s = df[(df['block']=='block7')&(df['model']==mdl)&(df['metric']==met)&(df['horizon_h']==h)]
            row[f'{met}_T+{h}'] = round(s['value'].mean(), 2)
    rows_s.append(row)

summary = pd.DataFrame(rows_s).set_index('Model')
mae_c  = [c for c in summary.columns if 'MAE' in c]
rmse_c = [c for c in summary.columns if 'RMSE' in c]
r2_c   = [c for c in summary.columns if 'R2' in c]
mape_c = [c for c in summary.columns if 'MAPE' in c]

summary.style \
    .background_gradient(cmap='RdYlGn_r', subset=mae_c) \
    .background_gradient(cmap='RdYlGn_r', subset=rmse_c) \
    .background_gradient(cmap='RdYlGn',   subset=r2_c) \
    .background_gradient(cmap='RdYlGn_r', subset=mape_c) \
    .format('{:.2f}')

---
## 12. Bảng chi tiết — Từng block × region

In [ ]:
for blk in BLOCKS:
    for reg in REGIONS:
        print(f'\n{"="*70}')
        print(f'  {blk} — {reg.upper()}')
        print(f'{"="*70}')
        rows_d = []
        for mdl in MODELS:
            row = {'Model': mdl}
            for h in H_NUMS:
                for met in ['MAE','RMSE','R2','MAPE']:
                    v = gv(blk, reg, met, mdl, h)
                    row[f'{met}_T+{h}'] = round(v,2) if not np.isnan(v) else '-'
            rows_d.append(row)
        detail = pd.DataFrame(rows_d).set_index('Model')
        display(detail)